<a href="https://colab.research.google.com/github/DHRUVCHARNE/AI-Learn-Notebooks/blob/main/bigram_MultiAttentionHead.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 256
n_head = 16
n_layer = 6
dropout = 0.2
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('/content/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out
class Head(nn.Module):
   """ One head of self attention"""
   def __init__(self,head_size):
      super().__init__()
      self.key = nn.Linear(n_embd,head_size,bias=False)
      self.query = nn.Linear(n_embd,head_size,bias=False)
      self.value = nn.Linear(n_embd,head_size,bias=False)
      self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
   def forward(self,x):
      B,T,C = x.shape
      k = self.key(x)  # (B,T,C)
      q= self.query(x) # (B,T,C)
      # Compute Attention Scores ("affinities")
      wei = q @ k.transpose(-2,-1) *  C**-0.5 # (B,T,C) @ (B,C,T) -> (B,T,T)
      wei = wei.masked_fill(self.tril[:T,:T]==0,float('-inf')) # (B,T,T)
      wei=F.softmax(wei,dim=-1) # (B,T,T)
      # perform the weighted aggregation of values
      v = self.value(x)
      out = wei @ v # (B,T,T) @ (B,T,C) = (B,T,C)
      return out

class MultiHeadAttention(nn.Module):
   """ Multiple Heads of self-attention in parallel"""
   def __init__(self,num_heads,head_size):
      super().__init__()
      self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
      self.proj=nn.Linear(n_embd,n_embd)
   def forward(self,x):
      out= torch.cat([h(x) for h in self.heads],dim=-1)
      out=self.proj(out)
      return out
class FeedForward(nn.Module):
  """ A Simple linear layer followed by non linearity"""
  def __init__(self,n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd,4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd,n_embd),
    )
  def forward(self,x):
        return self.net(x)
class Block(nn.Module):
  """ Transformer Block: communication followed by computation"""

  def __init__(self,n_embd,n_head):
    super().__init__()
    head_size=n_embd//n_head
    self.sa = MultiHeadAttention(n_head,head_size)
    self.ffwd = FeedForward(n_embd)
  def forward(self,x):
    x=x+self.sa(x)
    x=x+self.ffwd(x)
    return x
class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    # each token directly reads off the logits for the next token
    # from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size,n_embd)
    self.positional_embedding_table = nn.Embedding(block_size,n_embd)
    # self.sa_heads = MultiHeadAttention(16,n_embd//n_head) # ie 16 heads of 16 dimensional self attention
    # self.ffwd = FeedForward(n_embd)
    self.blocks = nn.Sequential(
        Block(n_embd,n_head=16),
        Block(n_embd,n_head=16),
        Block(n_embd,n_head=16)
    )
    self.lm_head = nn.Linear(n_embd,vocab_size)
  def forward(self,idx,targets=None):
    B,T = idx.shape
    # idx and targets are both (B,T) tensor of integers
    tok_embd=self.token_embedding_table(idx) #(B,T,C)
    pos_emb = self.positional_embedding_table(torch.arange(T,device=device)) #(T,C)
    x = tok_embd+pos_emb # (B,T,C)  -
    x = self.blocks(x) # Apply one head of self attention
    logits = self.lm_head(x) # (B,T,vocab_size)
    if targets is None:
        loss = None
    else:
        B,T,C = logits.shape
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)
    return logits,loss
  def generate(self,idx,max_new_tokens):
    # idx is (B,T) array of indices of the current context
    for _ in range(max_new_tokens):
      # crop context if needed
      idx_cond = idx[:, -block_size:]
      # get predictions
      logits,loss = self(idx_cond)
      # focus only on the last time step
      logits = logits[:,-1,:] # becomes (B,C)
      # apply softmax to the probabilities
      probs = F.softmax(logits,dim=-1) # (B,C)
      # sample from the distribution
      idx_next = torch.multinomial(probs,num_samples=1) # (B,1)
      # append the sampled index to the running sequence
      idx = torch.cat((idx,idx_next),dim=1) # (B,T+1)
    return idx
model=BigramLanguageModel()
m=model.to(device)
# create a pytorch optimizer
optimizer= torch.optim.AdamW(m.parameters(),lr=learning_rate)

for iter in range(max_iters):
   # Every once in a while evaluate the loss on train and eval sets
   if iter % eval_interval == 0:
      losses = estimate_loss()
      print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    # sample a batch of data
   xb,yb = get_batch('train')
   # evaluate the loss
   logits,loss = model(xb,yb)
   optimizer.zero_grad(set_to_none=True)
   loss.backward()
   optimizer.step()
# Generate from the model
context = torch.zeros((1,1),dtype=torch.long,device=device)
print(decode(m.generate(context,max_new_tokens=500)[0].tolist()))



step 0: train loss 4.5888, val loss 4.5756
step 500: train loss 2.1309, val loss 2.1791
step 1000: train loss 1.6979, val loss 1.8664
step 1500: train loss 1.5479, val loss 1.7471
step 2000: train loss 1.4694, val loss 1.6956
step 2500: train loss 1.4097, val loss 1.6472
step 3000: train loss 1.3739, val loss 1.6226
step 3500: train loss 1.3478, val loss 1.6260
step 4000: train loss 1.3203, val loss 1.6196
step 4500: train loss 1.2922, val loss 1.5965

Whip thy brows will? O lo, blessed the way, we'll
graves the change art that us hath bed hedge!
Hath art thy hearts are zoton,
Yourself forbid in Varlet in that mirtess,
Is not time in overty and villain:
And just, less doth him husband him speak;
Or she why hell make the patele down more:
The gatest of thou doth prizes to them,
Will cheeks now part of his guilt and thrust,
Is are great whither I learn, to fight?

KING EDWARD IV:
Brother, and day Warwick to Binind Nekep,
And king to change can y
